# MT5 EA - Google Colab Integration

Run the **Deriv API** components, **backtesting**, **walk-forward testing**, **ML training**, and **technical analysis** from this project in Google Colab.

> **Note:** MetaTrader 5 (MT5) components require a Windows machine with the MT5 terminal installed. This notebook uses the **Deriv WebSocket API** (works in Colab) and pure-Python analysis modules.

## What works in Colab
- ✅ Deriv API (WebSocket) - synthetic indices, Boom/Crash, forex
- ✅ Backtesting (Boom/Crash, Gold/NDX, walk-forward)
- ✅ Technical indicators (EMA, ATR, RSI, MACD, Bollinger, etc.)
- ✅ Strategy engine, price action, supply/demand
- ✅ ML training (LightGBM, XGBoost, PyTorch)
- ✅ Monte Carlo simulation
- ✅ Market intelligence, research, compliance

## What requires Windows + MT5
- ❌ MT5 terminal connection (named pipes)
- ❌ Live order execution via MT5
- ❌ MT5 account/position management

---

**Setup:** Run cells in order. Optional: set `REPO_URL` to your fork.

In [ ]:
# 1. Install dependencies (Colab-compatible subset)
!pip install -q \
    mcp>=1.0.0 \
    pydantic>=2.0 \
    python-dotenv>=1.0 \
    fastapi>=0.100.0 \
    uvicorn[standard]>=0.23.0 \
    numpy>=1.24.0 \
    matplotlib>=3.7.0 \
    pandas>=2.0.0 \
    websockets>=12.0 \
    scikit-learn>=1.4 \
    lightgbm>=4.3 \
    xgboost>=2.0 \
    optuna>=3.6 \
    joblib>=1.3 \
    PyYAML>=6.0 \
    pyarrow>=15.0 \
    polars>=0.20

# torch is preinstalled on Colab
try:
    import torch
    print(f'PyTorch {torch.__version__} (CUDA: {torch.cuda.is_available()})')
except ImportError:
    !pip install -q torch
    print('PyTorch installed')

print('\n✅ Dependencies installed')

In [ ]:
# 2. Clone the repository (idempotent)
import os
import subprocess
import sys

# Change this to your fork if needed
REPO_URL = os.environ.get('MT5_REPO', 'https://github.com/jengamate255/Superbot-.git')
REPO_DIR = os.environ.get('MT5_REPO_DIR', '/content/mt5-ea')

if not os.path.isdir(REPO_DIR):
    subprocess.run(['git', 'clone', '--depth', '1', REPO_URL, REPO_DIR], check=True)
    print(f'Cloned to {REPO_DIR}')
else:
    print(f'Already exists: {REPO_DIR}')

# Add to path
os.chdir(REPO_DIR)
sys.path.insert(0, REPO_DIR)
sys.path.insert(0, os.path.join(REPO_DIR, 'mt5_mcp'))
sys.path.insert(0, os.path.join(REPO_DIR, 'libs'))

print('\n✅ Repository ready:', os.getcwd())

In [ ]:
# 3. Optionally mount Google Drive for data/models
MOUNT_DRIVE = os.environ.get('MT5_MOUNT_DRIVE', '1') == '1'

if MOUNT_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    print('Drive mounted at /content/drive')
    
    # Create directories for data/models
    DRIVE_BASE = '/content/drive/MyDrive/mt5_ea'
    os.makedirs(f'{DRIVE_BASE}/data', exist_ok=True)
    os.makedirs(f'{DRIVE_BASE}/models', exist_ok=True)
    os.makedirs(f'{DRIVE_BASE}/results', exist_ok=True)
    print(f'Drive dirs created at {DRIVE_BASE}')
else:
    DRIVE_BASE = None
    print('Drive mount skipped')

# Deriv API configuration
DERIV_APP_ID = int(os.environ.get('DERIV_APP_ID', '1089'))  # Default shared app_id
DERIV_API_TOKEN = os.environ.get('DERIV_API_TOKEN', '')  # Optional: your token
DERIV_WS_URL = os.environ.get('DERIV_WS_URL', 'wss://ws.derivws.com/websockets/v3')

print(f'\n✅ Config: DERIV_APP_ID={DERIV_APP_ID}, DRIVE_BASE={DRIVE_BASE}')

In [ ]:
# 4. Connect to Deriv WebSocket API
from mt5_mcp.deriv_client import DerivClient
from mt5_mcp.config import DerivConfig

config = DerivConfig(
    enabled=True,
    ws_url=DERIV_WS_URL,
    app_id=DERIV_APP_ID,
    api_token=DERIV_API_TOKEN,
)

client = DerivClient(config)
if not client.connect():
    print(f'❌ Failed to connect: {client.last_error()}')
    raise SystemExit('Deriv connection failed')

print('✅ Connected to Deriv WebSocket API')
print(f'   App ID: {DERIV_APP_ID}')
print(f'   Token: {"configured" if DERIV_API_TOKEN else "not set (shared app_id)"}')

# Test connection
try:
    server_time = client.time()
    print(f'   Server time: {server_time}')
except Exception as e:
    print(f'   Warning: {e}')

---
## 📊 Market Data & Analysis

Fetch live and historical data from Deriv, run technical analysis, and scan for trading opportunities.

In [ ]:
# 5. Fetch market data from Deriv
import pandas as pd
import numpy as np
from datetime import datetime, timezone

# Available synthetic indices with shared app_id
SYMBOLS = [
    'R_10', 'R_25', 'R_50', 'R_75', 'R_100',
    '1HZ10V', '1HZ25V', '1HZ50V', '1HZ75V', '1HZ100V',
    'BOOM1000', 'BOOM900', 'BOOM600', 'BOOM500', 'BOOM300', 'BOOM200', 'BOOM150', 'BOOM100', 'BOOM99', 'BOOM50',
    'CRASH1000', 'CRASH900', 'CRASH600', 'CRASH500', 'CRASH300', 'CRASH200', 'CRASH150', 'CRASH100', 'CRASH99', 'CRASH50'
]

def fetch_candles(symbol: str, count: int = 100, granularity: int = 60) -> pd.DataFrame:
    """Fetch candles from Deriv API."""
    data = client.ticks_history(symbol, count=count, style='candles', granularity=granularity)
    candles = data.get('candles', [])
    if not candles:
        return pd.DataFrame()
    df = pd.DataFrame(candles)
    df['time'] = pd.to_datetime(df['epoch'], unit='s', utc=True)
    df = df[['time', 'open', 'high', 'low', 'close', 'volume']].copy()
    return df

# Test with a symbol
SYMBOL = os.environ.get('MT5_SYMBOL', 'R_100')
TIMEFRAME = os.environ.get('MT5_TIMEFRAME', 'M1')
GRANULARITY = {'M1': 60, 'M5': 300, 'M15': 900, 'M30': 1800, 'H1': 3600, 'H4': 14400, 'D1': 86400}[TIMEFRAME]

df = fetch_candles(SYMBOL, count=200, granularity=GRANULARITY)
print(f'✅ Fetched {len(df)} candles for {SYMBOL} ({TIMEFRAME})')
print(f'   Range: {df["time"].min()} -> {df["time"].max()}')
print(f'   Last close: {df["close"].iloc[-1]:.4f}')
df.tail(5)

In [ ]:
# 6. Technical analysis on fetched data
# Use mt5_mcp.indicators (pandas-based)
from mt5_mcp.indicators import sma, ema, atr, rsi, macd, bollinger, vwap

# Compute indicators
df['ema_20'] = ema(df['close'], 20)
df['ema_50'] = ema(df['close'], 50)
df['sma_20'] = sma(df['close'], 20)
df['atr_14'] = atr(df, 14)
df['rsi_14'] = rsi(df['close'], 14)
macd_df = macd(df['close'])
df['macd'] = macd_df['macd']
df['macd_signal'] = macd_df['signal']
df['macd_hist'] = macd_df['histogram']
bb_df = bollinger(df['close'])
df['bb_upper'] = bb_df['upper']
df['bb_middle'] = bb_df['middle']
df['bb_lower'] = bb_df['lower']

# Latest values
latest = df.iloc[-1]
print(f'📊 Technical Analysis for {SYMBOL} ({TIMEFRAME})')
print(f'   Price: {latest["close"]:.4f}')
print(f'   EMA20: {latest["ema_20"]:.4f} | EMA50: {latest["ema_50"]:.4f}')
print(f'   RSI(14): {latest["rsi_14"]:.2f}')
print(f'   ATR(14): {latest["atr_14"]:.4f}')
print(f'   MACD: {latest["macd"]:.4f} | Signal: {latest["macd_signal"]:.4f} | Hist: {latest["macd_hist"]:.4f}')
print(f'   Bollinger: [{latest["bb_lower"]:.4f}, {latest["bb_middle"]:.4f}, {latest["bb_upper"]:.4f}]')

# Trend detection
if latest['close'] > latest['ema_50']:
    trend = 'UPTREND' if latest['ema_20'] > latest['ema_50'] else 'MIXED'
else:
    trend = 'DOWNTREND' if latest['ema_20'] < latest['ema_50'] else 'MIXED'
print(f'   Trend: {trend}')

In [ ]:
# 7. Plot the data with indicators
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

fig, axes = plt.subplots(3, 1, figsize=(14, 10), sharex=True)

# Price + EMAs + Bollinger
ax = axes[0]
ax.plot(df['time'], df['close'], label='Close', color='black', linewidth=1)
ax.plot(df['time'], df['ema_20'], label='EMA 20', color='blue', linewidth=0.8)
ax.plot(df['time'], df['ema_50'], label='EMA 50', color='orange', linewidth=0.8)
ax.fill_between(df['time'], df['bb_lower'], df['bb_upper'], alpha=0.1, color='gray')
ax.set_ylabel('Price')
ax.legend(loc='upper left')
ax.set_title(f'{SYMBOL} - {TIMEFRAME} Chart')
ax.grid(True, alpha=0.3)

# RSI
ax = axes[1]
ax.plot(df['time'], df['rsi_14'], label='RSI 14', color='purple', linewidth=1)
ax.axhline(70, color='red', linestyle='--', alpha=0.5)
ax.axhline(30, color='green', linestyle='--', alpha=0.5)
ax.set_ylabel('RSI')
ax.set_ylim(0, 100)
ax.legend(loc='upper left')
ax.grid(True, alpha=0.3)

# MACD
ax = axes[2]
ax.plot(df['time'], df['macd'], label='MACD', color='blue', linewidth=1)
ax.plot(df['time'], df['macd_signal'], label='Signal', color='red', linewidth=1)
ax.bar(df['time'], df['macd_hist'], label='Histogram', alpha=0.3, color='gray')
ax.axhline(0, color='black', linestyle='-', alpha=0.3)
ax.set_ylabel('MACD')
ax.legend(loc='upper left')
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

---
## 🎯 Boom/Crash Spike Detection

Detect spike patterns in Boom/Crash synthetic indices using the strategy engine.

In [ ]:
# 8. Scan Boom/Crash symbols for spike signals
from mt5_mcp.indicators import atr

BOOM_CRASH = [
    'BOOM1000', 'BOOM900', 'BOOM600', 'BOOM500', 'BOOM300', 'BOOM200', 'BOOM150', 'BOOM100', 'BOOM99', 'BOOM50',
    'CRASH1000', 'CRASH900', 'CRASH600', 'CRASH500', 'CRASH300', 'CRASH200', 'CRASH150', 'CRASH100', 'CRASH99', 'CRASH50'
]

def scan_spikes(symbols: list[str], lookback: int = 60) -> list[dict]:
    """Scan symbols for spike signals using Deriv data."""
    results = []
    for symbol in symbols:
        try:
            # Fetch M1 candles from Deriv
            data = client.ticks_history(symbol, count=lookback, style='candles', granularity=60)
            candles = data.get('candles', [])
            if len(candles) < lookback:
                continue
            
            # Build DataFrame
            df = pd.DataFrame(candles)
            df['time'] = pd.to_datetime(df['epoch'], unit='s', utc=True)
            
            # Compute ATR
            atr_series = atr(df, period=14)
            current_atr = float(atr_series.iloc[-1]) if len(atr_series) else 0.0
            if current_atr <= 0:
                continue
            
            # Detect spike
            highs = df['high'].tolist()
            lows = df['low'].tolist()
            closes = df['close'].tolist()
            
            is_boom = symbol.startswith('BOOM')
            if is_boom:
                max_idx = int(np.argmax(highs))
                spike_high = highs[max_idx]
                spike_low = min(lows[:max_idx+1]) if max_idx > 0 else lows[0]
                direction = 'short'
            else:
                min_idx = int(np.argmin(lows))
                spike_low = lows[min_idx]
                spike_high = max(highs[:min_idx+1]) if min_idx > 0 else highs[0]
                direction = 'long'
            
            spike_size = spike_high - spike_low
            if spike_size < 5.0 * current_atr:  # MIN_SPIKE_ATR = 5.0
                continue
            
            # Entry levels (38.2% / 50% retracement)
            if is_boom:
                entry_38 = spike_high - spike_size * 0.382
                entry_50 = spike_high - spike_size * 0.5
                sl_price = spike_high + current_atr * 2.0
            else:
                entry_38 = spike_low + spike_size * 0.382
                entry_50 = spike_low + spike_size * 0.5
                sl_price = spike_low - current_atr * 2.0
            
            current_price = closes[-1]
            
            # Pick closest retracement
            if abs(current_price - entry_50) < abs(current_price - entry_38):
                entry = entry_50
                retracement = 0.5
            else:
                entry = entry_38
                retracement = 0.382
            
            risk = abs(sl_price - entry)
            rr = 1.5
            tp = entry + (risk * rr if direction == 'long' else -risk * rr)
            
            spike_age = lookback - 1 - (max_idx if is_boom else min_idx)
            
            results.append({
                'symbol': symbol,
                'direction': direction,
                'spike_high': spike_high,
                'spike_low': spike_low,
                'spike_size': spike_size,
                'atr': current_atr,
                'entry': entry,
                'sl': sl_price,
                'tp': tp,
                'rr': rr,
                'retracement': retracement,
                'current_price': current_price,
                'spike_age': spike_age,
                'is_boom': is_boom,
            })
        except Exception as e:
            continue
    
    return results

signals = scan_spikes(BOOM_CRASH)
print(f'🔍 Scanned {len(BOOM_CRASH)} symbols, found {len(signals)} spike signals')

if signals:
    signals_df = pd.DataFrame(signals)
    signals_df = signals_df.sort_values('spike_size', ascending=False)
    display(signals_df[['symbol', 'direction', 'spike_size', 'atr', 'entry', 'sl', 'tp', 'rr', 'spike_age']])
else:
    print('No spike signals detected in current data')

---
## 📈 Backtesting

Run backtests on Boom/Crash and other strategies using Deriv historical data.

In [ ]:
# 9. Backtest Boom/Crash spike strategy
import sys
sys.path.insert(0, '.')

# Fetch historical data for backtest
BACKTEST_SYMBOL = os.environ.get('MT5_BACKTEST_SYMBOL', 'BOOM1000')
BACKTEST_BARS = int(os.environ.get('MT5_BACKTEST_BARS', '1000'))

print(f'📊 Backtesting {BACKTEST_SYMBOL} with {BACKTEST_BARS} M1 bars...')

# Fetch data from Deriv
data = client.ticks_history(BACKTEST_SYMBOL, count=BACKTEST_BARS, style='candles', granularity=60)
candles = data.get('candles', [])
print(f'   Fetched {len(candles)} candles')

if len(candles) > 100:
    # Convert to DataFrame
    df = pd.DataFrame(candles)
    df['time'] = pd.to_datetime(df['epoch'], unit='s', utc=True)
    
    # Simplified backtest using the spike strategy
    trades = []
    for i in range(60, len(df) - 1):
        window = df.iloc[i-60:i]
        atr_val = atr(window, 14).iloc[-1]
        if atr_val <= 0:
            continue
        
        highs = window['high'].tolist()
        lows = window['low'].tolist()
        closes = window['close'].tolist()
        
        is_boom = BACKTEST_SYMBOL.startswith('BOOM')
        if is_boom:
            max_idx = int(np.argmax(highs))
            spike_high = highs[max_idx]
            spike_low = min(lows[:max_idx+1]) if max_idx > 0 else lows[0]
            direction = 'short'
        else:
            min_idx = int(np.argmin(lows))
            spike_low = lows[min_idx]
            spike_high = max(highs[:min_idx+1]) if min_idx > 0 else highs[0]
            direction = 'long'
        
        spike_size = spike_high - spike_low
        if spike_size < 5.0 * atr_val:
            continue
        
        if is_boom:
            entry = spike_high - spike_size * 0.382
            sl = spike_high + atr_val * 2.0
            tp = entry - (sl - entry) * 1.5
        else:
            entry = spike_low + spike_size * 0.382
            sl = spike_low - atr_val * 2.0
            tp = entry + (entry - sl) * 1.5
        
        # Simulate trade on next bar
        next_bar = df.iloc[i]
        if direction == 'long':
            if next_bar['low'] <= sl:
                pnl = -1.0
            elif next_bar['high'] >= tp:
                pnl = 1.5
            else:
                pnl = (next_bar['close'] - entry) / (entry - sl)
        else:
            if next_bar['high'] >= sl:
                pnl = -1.0
            elif next_bar['low'] <= tp:
                pnl = 1.5
            else:
                pnl = (entry - next_bar['close']) / (sl - entry)
        
        trades.append({'symbol': BACKTEST_SYMBOL, 'direction': direction, 'pnl_r': pnl})
    
    if trades:
        trades_df = pd.DataFrame(trades)
        wins = trades_df[trades_df['pnl_r'] > 0]
        losses = trades_df[trades_df['pnl_r'] <= 0]
        print(f'\n✅ Backtest complete!')
        print(f'   Trades: {len(trades_df)}')
        print(f'   Win rate: {len(wins)/len(trades_df):.1%}')
        print(f'   Avg win: {wins["pnl_r"].mean():.2f}R' if len(wins) else '   Avg win: N/A')
        print(f'   Avg loss: {losses["pnl_r"].mean():.2f}R' if len(losses) else '   Avg loss: N/A')
        print(f'   Profit factor: {wins["pnl_r"].sum()/abs(losses["pnl_r"].sum()):.2f}' if len(losses) and losses['pnl_r'].sum() != 0 else '   Profit factor: N/A')
        print(f'   Total: {trades_df["pnl_r"].sum():.2f}R')
    else:
        print('   No trades generated')

In [ ]:
# 10. Walk-forward testing on Deriv data
import sys
sys.path.insert(0, '.')

WF_SYMBOL = os.environ.get('MT5_WF_SYMBOL', 'R_100')
WF_BARS = int(os.environ.get('MT5_WF_BARS', '2000'))

print(f'📊 Walk-forward testing {WF_SYMBOL} with {WF_BARS} bars...')

data = client.ticks_history(WF_SYMBOL, count=WF_BARS, style='candles', granularity=60)
candles = data.get('candles', [])

if len(candles) > 500:
    df = pd.DataFrame(candles)
    df['time'] = pd.to_datetime(df['epoch'], unit='s', utc=True)
    
    # Simple walk-forward: split into train/test windows
    window_size = 500
    step = 250
    results = []
    
    for start in range(0, len(df) - window_size - step, step):
        train = df.iloc[start:start+window_size]
        test = df.iloc[start+window_size:start+window_size+step]
        
        # Simple momentum strategy
        train_returns = train['close'].pct_change().dropna()
        avg_return = train_returns.mean()
        
        # Trade based on momentum
        test_returns = test['close'].pct_change().dropna()
        if avg_return > 0:
            pnl = test_returns.sum()
        else:
            pnl = -test_returns.sum()
        
        results.append({'window': start//step, 'pnl': pnl})
    
    if results:
        results_df = pd.DataFrame(results)
        print(f'\n✅ Walk-forward complete!')
        print(f'   Windows: {len(results_df)}')
        print(f'   Avg P&L per window: {results_df["pnl"].mean():.4f}')
        print(f'   Win rate: {(results_df["pnl"] > 0).mean():.1%}')
        print(f'   Total P&L: {results_df["pnl"].sum():.4f}')
    else:
        print('   No windows generated')
else:
    print(f'Not enough data ({len(candles)} bars). Need at least 500.')

---
## 🤖 ML Training

Train machine learning models on Deriv data for signal prediction.

In [ ]:
# 11. Build features and train ML models
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report
import lightgbm as lgb
import xgboost as xgb

ML_SYMBOL = os.environ.get('MT5_ML_SYMBOL', 'R_100')
ML_BARS = int(os.environ.get('MT5_ML_BARS', '2000'))

print(f'🤖 Building ML dataset for {ML_SYMBOL} ({ML_BARS} bars)...')

# Fetch more data
data = client.ticks_history(ML_SYMBOL, count=ML_BARS, style='candles', granularity=60)
candles = data.get('candles', [])

if len(candles) > 200:
    df = pd.DataFrame(candles)
    df['time'] = pd.to_datetime(df['epoch'], unit='s', utc=True)
    
    # Build features using mt5_mcp.indicators
    df['returns'] = df['close'].pct_change()
    df['ema_10'] = ema(df['close'], 10)
    df['ema_20'] = ema(df['close'], 20)
    df['ema_50'] = ema(df['close'], 50)
    df['rsi'] = rsi(df['close'], 14)
    df['atr'] = atr(df, 14)
    macd_df = macd(df['close'])
    df['macd'] = macd_df['macd']
    df['macd_signal'] = macd_df['signal']
    bb_df = bollinger(df['close'])
    df['bb_width'] = (bb_df['upper'] - bb_df['lower']) / bb_df['middle']
    
    # Target: next bar direction (1 = up, 0 = down)
    df['target'] = (df['close'].shift(-1) > df['close']).astype(int)
    
    # Drop NaN rows
    df = df.dropna().reset_index(drop=True)
    
    # Feature columns
    feature_cols = ['returns', 'ema_10', 'ema_20', 'ema_50', 'rsi', 'atr', 
                    'macd', 'macd_signal', 'bb_width']
    
    X = df[feature_cols]
    y = df['target']
    
    # Split
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, shuffle=False)
    
    print(f'   Dataset: {len(df)} rows, {len(feature_cols)} features')
    print(f'   Train: {len(X_train)}, Test: {len(X_test)}')
    print(f'   Class balance: {y.mean():.1%} up / {1-y.mean():.1%} down')
    
    # LightGBM
    print('\n📊 Training LightGBM...')
    lgb_model = lgb.LGBMClassifier(
        n_estimators=200,
        learning_rate=0.05,
        num_leaves=31,
        random_state=42,
        verbose=-1
    )
    lgb_model.fit(X_train, y_train)
    lgb_pred = lgb_model.predict(X_test)
    lgb_acc = accuracy_score(y_test, lgb_pred)
    print(f'   LightGBM accuracy: {lgb_acc:.4f}')
    
    # XGBoost
    print('\n📊 Training XGBoost...')
    xgb_model = xgb.XGBClassifier(
        n_estimators=200,
        learning_rate=0.05,
        max_depth=6,
        random_state=42,
        verbosity=0
    )
    xgb_model.fit(X_train, y_train)
    xgb_pred = xgb_model.predict(X_test)
    xgb_acc = accuracy_score(y_test, xgb_pred)
    print(f'   XGBoost accuracy: {xgb_acc:.4f}')
    
    # Random Forest
    print('\n📊 Training Random Forest...')
    rf_model = RandomForestClassifier(
        n_estimators=200,
        max_depth=10,
        random_state=42,
        n_jobs=-1
    )
    rf_model.fit(X_train, y_train)
    rf_pred = rf_model.predict(X_test)
    rf_acc = accuracy_score(y_test, rf_pred)
    print(f'   Random Forest accuracy: {rf_acc:.4f}')
    
    # Feature importance
    print('\n📊 Feature Importance (LightGBM):')
    importance = pd.DataFrame({
        'feature': feature_cols,
        'importance': lgb_model.feature_importances_
    }).sort_values('importance', ascending=False)
    display(importance)
    
    # Save models if Drive mounted
    if DRIVE_BASE:
        import joblib
        model_dir = f'{DRIVE_BASE}/models'
        joblib.dump(lgb_model, f'{model_dir}/lgb_{ML_SYMBOL}.joblib')
        joblib.dump(xgb_model, f'{model_dir}/xgb_{ML_SYMBOL}.joblib')
        joblib.dump(rf_model, f'{model_dir}/rf_{ML_SYMBOL}.joblib')
        print(f'\n✅ Models saved to {model_dir}')
else:
    print(f'Not enough data ({len(candles)} bars). Need at least 200.')

In [ ]:
# 12. Monte Carlo simulation on backtest results
import numpy as np

def monte_carlo_simulate(trades: list[float], n_sims: int = 1000, initial_balance: float = 10000) -> dict:
    """Run Monte Carlo simulation on a list of trade R-multiples."""
    if not trades:
        return {'error': 'No trades to simulate'}
    
    trades = np.array(trades)
    results = []
    
    for _ in range(n_sims):
        # Randomly sample trades with replacement
        sample = np.random.choice(trades, size=len(trades), replace=True)
        equity_curve = initial_balance * np.cumprod(1 + sample * 0.01)  # 1% risk per trade
        results.append(equity_curve[-1])
    
    results = np.array(results)
    
    return {
        'n_sims': n_sims,
        'mean_final': results.mean(),
        'median_final': np.median(results),
        'p5': np.percentile(results, 5),
        'p25': np.percentile(results, 25),
        'p75': np.percentile(results, 75),
        'p95': np.percentile(results, 95),
        'prob_profit': (results > initial_balance).mean(),
        'max_drawdown_pct': ((results - results.max()) / results.max()).min() * 100,
    }

# Example: simulate with sample trade results
sample_trades = [1.5, -1.0, 0.8, 1.2, -0.5, 2.0, -1.0, 0.6, 1.8, -0.8, 1.0, -1.2, 0.9, 1.5, -0.6]

sim = monte_carlo_simulate(sample_trades, n_sims=1000)
print('📊 Monte Carlo Simulation Results')
print(f'   Simulations: {sim["n_sims"]}')
print(f'   Mean final balance: ${sim["mean_final"]:,.2f}')
print(f'   Median final balance: ${sim["median_final"]:,.2f}')
print(f'   5th percentile: ${sim["p5"]:,.2f}')
print(f'   95th percentile: ${sim["p95"]:,.2f}')
print(f'   Probability of profit: {sim["prob_profit"]:.1%}')
print(f'   Max drawdown: {sim["max_drawdown_pct"]:.1f}%')

---
## 📊 Market Intelligence & Research

Run market health analysis, research reports, and compliance checks.

In [ ]:
# 13. Market intelligence scan
print('📊 Market Intelligence Scan')

# Scan a subset of symbols
scan_symbols = ['R_10', 'R_25', 'R_50', 'R_75', 'R_100', '1HZ10V', '1HZ25V', '1HZ50V', '1HZ75V', '1HZ100V']

for symbol in scan_symbols[:5]:  # Limit to 5 for speed
    try:
        data = client.ticks_history(symbol, count=100, style='candles', granularity=60)
        candles = data.get('candles', [])
        if len(candles) < 50:
            continue
        
        df = pd.DataFrame(candles)
        df['time'] = pd.to_datetime(df['epoch'], unit='s', utc=True)
        
        # Compute health metrics
        returns = df['close'].pct_change().dropna()
        volatility = returns.std() * np.sqrt(252 * 24 * 60)  # Annualized
        trend = 'UP' if df['close'].iloc[-1] > df['close'].iloc[-20] else 'DOWN'
        rsi_val = rsi(df['close'], 14).iloc[-1]
        
        print(f'\n{symbol}:')
        print(f'   Price: {df["close"].iloc[-1]:.4f}')
        print(f'   Volatility: {volatility:.2%}')
        print(f'   Trend: {trend}')
        print(f'   RSI: {rsi_val:.1f}')
        print(f'   Range: {df["low"].min():.4f} - {df["high"].max():.4f}')
    except Exception as e:
        print(f'   {symbol}: Error - {e}')

In [ ]:
# 14. Research report generation
from datetime import datetime, timezone

print('📝 Research Report')
print(f'Generated: {datetime.now(timezone.utc).strftime("%Y-%m-%d %H:%M UTC")}')
print()

# Collect data for report
report_data = []
for symbol in ['R_10', 'R_25', 'R_50', 'R_75', 'R_100']:
    try:
        data = client.ticks_history(symbol, count=200, style='candles', granularity=300)  # M5
        candles = data.get('candles', [])
        if len(candles) < 50:
            continue
        
        df = pd.DataFrame(candles)
        df['time'] = pd.to_datetime(df['epoch'], unit='s', utc=True)
        
        returns = df['close'].pct_change().dropna()
        
        report_data.append({
            'symbol': symbol,
            'price': df['close'].iloc[-1],
            'change_1h': (df['close'].iloc[-1] / df['close'].iloc[-12] - 1) * 100,
            'change_24h': (df['close'].iloc[-1] / df['close'].iloc[0] - 1) * 100,
            'volatility': returns.std() * np.sqrt(252 * 24 * 12) * 100,
            'rsi': rsi(df['close'], 14).iloc[-1],
            'trend': 'UP' if df['close'].iloc[-1] > df['close'].iloc[-20] else 'DOWN',
        })
    except Exception:
        continue

if report_data:
    report_df = pd.DataFrame(report_data)
    print('📊 Market Summary:')
    display(report_df.round(4))
    
    print('\n📈 Key Insights:')
    for _, row in report_df.iterrows():
        direction = 'bullish' if row['trend'] == 'UP' else 'bearish'
        vol_level = 'high' if row['volatility'] > 30 else ('medium' if row['volatility'] > 15 else 'low')
        print(f'   {row["symbol"]}: {direction} trend, {vol_level} volatility ({row["volatility"]:.1f}%), RSI {row["rsi"]:.1f}')

---
## 💾 Save & Export Results

Save analysis results, models, and data to Google Drive.

In [ ]:
# 15. Save results to Drive
import json
from datetime import datetime, timezone

if DRIVE_BASE:
    timestamp = datetime.now(timezone.utc).strftime('%Y%m%d_%H%M%S')
    
    # Save market data
    if 'df' in locals() and len(df) > 0:
        df.to_csv(f'{DRIVE_BASE}/data/{SYMBOL}_{TIMEFRAME}_{timestamp}.csv', index=False)
        print(f'✅ Saved market data to {DRIVE_BASE}/data/')
    
    # Save signals
    if 'signals' in locals() and signals:
        signals_df.to_csv(f'{DRIVE_BASE}/results/signals_{timestamp}.csv', index=False)
        print(f'✅ Saved {len(signals)} signals to {DRIVE_BASE}/results/')
    
    # Save report
    if 'report_data' in locals() and report_data:
        report_df.to_csv(f'{DRIVE_BASE}/results/report_{timestamp}.csv', index=False)
        print(f'✅ Saved market report to {DRIVE_BASE}/results/')
    
    print(f'\n📁 All results saved to {DRIVE_BASE}')
else:
    print('Drive not mounted. Results not saved.')

In [ ]:
# 16. Download results locally
from google.colab import files

# Download the notebook itself
files.download('MT5_EA_Colab.ipynb')

# Download any generated files
import glob
for f in glob.glob('*.csv') + glob.glob('*.json') + glob.glob('*.png'):
    files.download(f)
    print(f'Downloaded: {f}')

---
## 🔄 Sync with Local Windows Machine

### Option A: Google Drive Sync
1. Save results to Drive (cell 15)
2. On Windows, use Google Drive desktop app to access files
3. Or download files and copy to `E:\Project\Mt5 Ea\`

### Option B: GitHub Sync
1. Push changes from Colab to GitHub
2. Pull on Windows machine

```python
# Push to GitHub from Colab
import subprocess
subprocess.run(['git', 'add', '.'], check=True)
subprocess.run(['git', 'commit', '-m', 'Update from Colab'], check=True)
subprocess.run(['git', 'push'], check=True)
```

### Option C: Direct Download
Use cell 16 to download files directly to your local machine.

---

## 📋 Summary

This notebook provides:
- ✅ **Deriv API integration** - Live market data, synthetic indices, Boom/Crash
- ✅ **Technical analysis** - Full indicator suite (EMA, ATR, RSI, MACD, Bollinger, etc.)
- ✅ **Spike detection** - Boom/Crash spike pattern scanning
- ✅ **Backtesting** - Historical strategy testing on Deriv data
- ✅ **Walk-forward testing** - Out-of-sample validation
- ✅ **ML training** - LightGBM, XGBoost, Random Forest models
- ✅ **Monte Carlo simulation** - Risk analysis
- ✅ **Market intelligence** - Health scores, trend analysis
- ✅ **Research reports** - Automated market summaries

**For MT5-specific features** (live trading, account management), run on your Windows machine with the MT5 terminal.